<a href="https://colab.research.google.com/github/yunhaod/postura/blob/main/posturaModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary libraries
!pip install pandas numpy scikit-learn tensorflow
!pip install tflite-support

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras

#lt rt ml mr bl br ir posture_classification

file_path = ''
df = pd.read_csv(file_path)
x_names = []
y_name = 'posture_status'
X = df[x_names]
y = df[y_name]

Xtr, Xts = train_test_split(X, test_size = 0.2)
ytr, yts = train_test_split(y, test_size = 0.2)


# Assume input shape is (7,) representing positional values of each pressure sensor
model = keras.Sequential([
    keras.layers.InputLayer(input_shape=(7,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(6, activation='softmax')  # 6 output classes for 6 types of posture status (good, neck slouching, spine slouch,
])                                              #l.                                                eft leaning, right leaning, everything bad)

# Compile the model - This was missing and is required before fitting
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#performance indiactes how accurate the model is
performance = np.zeros(5)
kf = KFold(n_splits=5, shuffle = True)

for i,(train_idx, val_idx) in enumerate(kf.split(Xtr,ytr)):
  x_train, x_validate = Xtr.iloc[train_idx], Xtr.iloc[val_idx]
  y_train, y_validate = ytr.iloc[train_idx], ytr.iloc[val_idx]
  model.fit(x_train, y_train, epochs=500, batch_size=16, validation_data=(x_validate, y_validate))
  performance[i] = model.evaluate(x_validate, y_validate)[1]

print(f"Performance across folds{performance.mean()}")

predictions = model.predict(Xts)
accuracy = r2_score(yts, predictions)
print(f"Accuracy: {accuracy}")


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]
tflite_model = converter.convert()

# Write TFLite model to a C source (or header) file
with open("posture_model + '.h', 'w') as file:
  file.write(hex_to_c_array(tflite_model, c_model_name))

# Save the TensorFlow Lite model
open("arduino_model.tflite", "wb").write(tflite_model)

#test if this section is needed
# Generate the header file
!echo "const unsigned char model[] = {" > /content/model.h
!cat arduino_model.tflite | xxd -i >> /content/model.h
!echo "};" >> /content/model.h






FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

In [ ]:
# Function: Convert some hex value into an array for C programming
def hex_to_c_array(hex_data, var_name):

  c_str = ''

  # Create header guard
  c_str += '#ifndef ' + var_name.upper() + '_H\n'
  c_str += '#define ' + var_name.upper() + '_H\n\n'

  # Add array length at top of file
  c_str += '\nunsigned int ' + var_name + '_len = ' + str(len(hex_data)) + ';\n'

  # Declare C variable
  c_str += 'unsigned char ' + var_name + '[] = {'
  hex_array = []
  for i, val in enumerate(hex_data) :

    # Construct string from hex
    hex_str = format(val, '#04x')

    # Add formatting so each line stays within 80 characters
    if (i + 1) < len(hex_data):
      hex_str += ','
    if (i + 1) % 12 == 0:
      hex_str += '\n '
    hex_array.append(hex_str)

  # Add closing brace
  c_str += '\n ' + format(' '.join(hex_array)) + '\n};\n\n'

  # Close out header guard
  c_str += '#endif //' + var_name.upper() + '_H'

  return c_str